# 文档加载器Docunment Loaders
## 1、加载txt

In [1]:

from langchain_community.document_loaders import TextLoader, CSVLoader

loader = TextLoader(
    file_path="../asset/load/01-langchain-utf-8.txt",
    encoding="utf-8",
)
docs = loader.load()
print(docs)

[Document(metadata={'source': '../asset/load/01-langchain-utf-8.txt'}, page_content='LangChain 是一个用于构建基于大语言模型（LLM）应用的开发框架，旨在帮助开发者更高效地集成、管理和增强大语言模型的能力，构建端到端的应用程序。它提供了一套模块化工具和接口，支持从简单的文本生成到复杂的多步骤推理任务')]


In [3]:

from langchain_community.document_loaders import  CSVLoader

loader01 = CSVLoader(
    file_path="../asset/load/02-load.csv",

)
docs01 = loader01.load()
print(docs01)

[Document(metadata={'source': '../asset/load/02-load.csv', 'row': 0}, page_content='id: 1\ntitle: Introduction to Python\ncontent: Python is a popular programming language.\nauthor: John Doe'), Document(metadata={'source': '../asset/load/02-load.csv', 'row': 1}, page_content='id: 2\ntitle: Data Science Basics\ncontent: Data science involves statistics and machine learning.\nauthor: Jane Smith'), Document(metadata={'source': '../asset/load/02-load.csv', 'row': 2}, page_content='id: 3\ntitle: Web Development\ncontent: HTML, CSS and JavaScript are core web technologies.\nauthor: Mike Johnson'), Document(metadata={'source': '../asset/load/02-load.csv', 'row': 3}, page_content='id: 4\ntitle: Artificial Intelligence\ncontent: AI is transforming many industries.\nauthor: Sarah Williams')]


In [ ]:
from  langchain_community .document_loaders import JSONLoader
from rich import print as rprint
# 2 .定义JSONLoader对象
# 情况1
json_loader=JSONLoader(
file_path="../asset/load/03-load.json",
jq_schema=".", ## 提取所有字段
text_content=False #保持原始JSON 结构，将提取的数据转换为JSON字符串存入page_content字段中
)
# 情况2
# .messages[].content:遍历.messages[]中所有元素从每一个元素中提取.content字段
# json_loader=JSONLoader(
#     file_path="../asset/load/03-load .json ",
#     jq_schema=".messages [].content "
# )
# 3 .加载
docs = json_loader .load()
rprint(docs)

In [4]:
import os
import time
import requests
from dotenv import load_dotenv

load_dotenv(override=True)

def upload_files(file_paths: list[str]) -> str:
    """批量上传文件"""
    url = "https://mineru.net/api/v4/file-urls/batch"
    api_token = os.getenv("MINERU_API_TOKEN")
    header = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {api_token}",
    }

    files_info = [
        {
            "name": os.path.basename(file_path),
            "is_ocr": True,
            "data_id": f"file_{i}",
        }
        for i, file_path in enumerate(file_paths)
    ]

    data = {
        "enable_formula": True,
        "enable_table": True,
        "language": "ch",
        "files": files_info,
    }

    try:
        response = requests.post(url, headers=header, json=data)
        if response.status_code == 200:
            result = response.json()
            print("response success. result:{}".format(result))

            if result["code"] == 0:
                batch_id = result["data"]["batch_id"]
                urls = result["data"]["file_urls"]
                print("batch_id:{}\nurls:{}".format(batch_id, urls))

                for i in range(0, len(urls)):
                    with open(file_paths[i], "rb") as f:
                        res_upload = requests.put(urls[i], data=f)
                        if res_upload.status_code == 200:
                            print(f"{urls[i]} upload success")
                        else:
                            print(f"{urls[i]} upload failed")
                            return None

                return batch_id
            else:
                print("apply upload url failed, reason: {}".format(result.get("msg")))
                return None
        else:
            print(
                "response not success. status:{}, result:{}".format(
                    response.status_code, response.text
                )
            )
            return None

    except Exception as err:
        print(err)
        return None


def download_files(batch_id):
    """批量获取任务结果"""
    if not batch_id:
        print("batch_id为空，跳过下载")
        return

    os.makedirs("parsed_files", exist_ok=True)

    url = f"https://mineru.net/api/v4/extract-results/batch/{batch_id}"
    api_token = os.getenv("MINERU_API_TOKEN")
    header = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {api_token}",
    }

    failed_files = set()
    done_files = set()

    while True:
        res = requests.get(url, headers=header)
        result_json = res.json()

        if res.status_code != 200 or result_json.get("code") != 0:
            print("get result failed:", result_json)
            break

        extract_results = result_json["data"]["extract_result"]

        for result in extract_results:
            data_id = result["data_id"]

            if result["state"] == "failed":
                failed_files.add(data_id)

            elif result["state"] == "done" and data_id not in done_files:
                done_files.add(data_id)

                full_zip_url = result["full_zip_url"]
                res_download = requests.get(full_zip_url, stream=True)

                with open(
                    f"parsed_files/{result['file_name']}_{result['data_id']}.zip", "wb"
                ) as f:
                    for chunk in res_download.iter_content(chunk_size=1024):
                        if chunk:
                            f.write(chunk)

        if len(failed_files) + len(done_files) == len(extract_results):
            break

        time.sleep(5)

    for i in failed_files:
        print("failed :", i)

    for i in done_files:
        print("done :", i)


if __name__ == "__main__":
    file_paths = ["../asset/load/04-sample.pdf"]
    batch_id = upload_files(file_paths)

    if batch_id:
        download_files(batch_id)

response success. result:{'code': 0, 'msg': 'ok', 'trace_id': '83124addd117c13431767dc14f98b261', 'data': {'batch_id': '30b7cf52-9b03-4b57-bbc3-d8576400f1a5', 'file_urls': ['https://mineru.oss-cn-shanghai.aliyuncs.com/api-upload/extract/2026-09-16/30b7cf52-9b03-4b57-bbc3-d8576400f1a5/0ecca95d-a941-4134-b248-dbccaec0b287.pdf?Expires=1789659964&OSSAccessKeyId=LTAI5t8fSGMgiRhQn4mpp926&Signature=EMidDyBWvqrdyeNaYhHeYVm8Hhk%3D']}}
batch_id:30b7cf52-9b03-4b57-bbc3-d8576400f1a5
urls:['https://mineru.oss-cn-shanghai.aliyuncs.com/api-upload/extract/2026-09-16/30b7cf52-9b03-4b57-bbc3-d8576400f1a5/0ecca95d-a941-4134-b248-dbccaec0b287.pdf?Expires=1789659964&OSSAccessKeyId=LTAI5t8fSGMgiRhQn4mpp926&Signature=EMidDyBWvqrdyeNaYhHeYVm8Hhk%3D']
https://mineru.oss-cn-shanghai.aliyuncs.com/api-upload/extract/2026-09-16/30b7cf52-9b03-4b57-bbc3-d8576400f1a5/0ecca95d-a941-4134-b248-dbccaec0b287.pdf?Expires=1789659964&OSSAccessKeyId=LTAI5t8fSGMgiRhQn4mpp926&Signature=EMidDyBWvqrdyeNaYhHeYVm8Hhk%3D upload succ